In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [3]:
import json

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
dataset = generate_dataset()
print(dataset)

with open('012_prompt_evals2OutputFileTest.json', 'w') as f:
    json.dump(dataset, f, indent=2)

[{'task': 'Write a Python function that validates an AWS S3 bucket name according to AWS naming rules (3-63 characters, lowercase letters, numbers, and hyphens only, must start and end with letter or number)'}, {'task': "Create a JSON object representing an AWS IAM policy that allows read-only access to a specific S3 bucket named 'my-data-bucket'"}, {'task': 'Write a regular expression that matches valid AWS ARN (Amazon Resource Name) formats for EC2 instances in the us-east-1 region'}]


In [5]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [6]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [7]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [ ]:
with open("012_prompt_evals2OutputFileTest.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:

print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 URI Parser\n\nHere's a comprehensive solution with multiple approaches:\n\n```python\nfrom typing import Tuple, Dict\nfrom urllib.parse import urlparse\n\ndef parse_s3_uri_simple(s3_uri: str) -> Tuple[str, str, str]:\n    \"\"\"\n    Parse an S3 URI and extract bucket name, folder path, and file name.\n    \n    Args:\n        s3_uri: Full S3 URI like 's3://my-bucket/folder/subfolder/file.txt'\n    \n    Returns:\n        Tuple of (bucket_name, folder_path, file_name)\n    \n    Raises:\n        ValueError: If the URI is not a valid S3 URI\n    \"\"\"\n    if not s3_uri.startswith('s3://'):\n        raise ValueError(f\"Invalid S3 URI: {s3_uri}. Must start with 's3://'\")\n    \n    # Remove 's3://' prefix\n    uri_without_scheme = s3_uri[5:]\n    \n    # Split bucket from the rest\n    parts = uri_without_scheme.split('/', 1)\n    bucket_name = parts[0]\n    \n    if not bucket_name:\n        raise ValueError(\"Bucket name is empty\")\n    \n    # If no pa